In [ ]:
# ==============================================================
# 06 – Single-Agent PPO for Credit Decisioning
# Strong single-agent baseline before Multi-Agent MARL
# Supports RQ1 and RQ4
# ==============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ROOT = Path(".")
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1. Load Fused Features
# --------------------------------------------------------------
X = np.load(DATA_PROCESSED / "X_train_fused.npy")
y = np.load(DATA_PROCESSED / "y_train.npy")
thin = np.load(DATA_PROCESSED / "thin_train.npy")

X_test = np.load(DATA_PROCESSED / "X_test_fused.npy")
y_test = np.load(DATA_PROCESSED / "y_test.npy")
thin_test = np.load(DATA_PROCESSED / "thin_test.npy")

state_dim = X.shape[1]
print(f"State dimension: {state_dim}")
print(f"Train samples: {len(X)} | Default rate: {y.mean():.2%}")

# --------------------------------------------------------------
# 2. Credit Environment (same design as notebook 04)
# --------------------------------------------------------------
class CreditDecisionEnv:
    def __init__(self, X, y, thin, cost_fn=5.0, cost_fp=1.0):
        self.X = X.astype(np.float32)
        self.y = y
        self.thin = thin
        self.cost_fn = cost_fn
        self.cost_fp = cost_fp
        self.n = len(X)
        self.action_dim = 3
        self.reset()

    def reset(self):
        self.indices = np.random.permutation(self.n)
        self.ptr = 0
        return self.X[self.indices[0]]

    def step(self, action):
        idx = self.indices[self.ptr]
        true_label = self.y[idx]
        is_thin = self.thin[idx]

        if action == 1:          # Approve
            if true_label == 0:
                reward = 1.0 + (0.35 if is_thin else 0.0)
            else:
                reward = -self.cost_fn
        elif action == 2:        # Counter-offer
            if true_label == 0:
                reward = 0.55
            else:
                reward = -self.cost_fn * 0.55
        else:                    # Reject
            if true_label == 1:
                reward = 0.45
            else:
                reward = -self.cost_fp - (0.25 if is_thin else 0.0)

        self.ptr += 1
        done = self.ptr >= self.n
        next_state = self.X[self.indices[self.ptr]] if not done else np.zeros(self.X.shape[1], dtype=np.float32)
        return next_state, reward, done

# --------------------------------------------------------------
# 3. Actor-Critic Network for PPO
# --------------------------------------------------------------
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim=3, hidden=128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU()
        )
        self.actor = nn.Linear(hidden, action_dim)
        self.critic = nn.Linear(hidden, 1)

    def forward(self, x):
        features = self.shared(x)
        logits = self.actor(features)
        value = self.critic(features).squeeze(-1)
        return logits, value

# --------------------------------------------------------------
# 4. PPO Hyperparameters
# --------------------------------------------------------------
ACTION_DIM = 3
LR = 3e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_EPS = 0.2
EPOCHS_PER_UPDATE = 4
BATCH_SIZE = 64
NUM_EPISODES = 60
UPDATE_TIMESTEPS = 2048

model = ActorCritic(state_dim, ACTION_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

env = CreditDecisionEnv(X, y, thin)

# --------------------------------------------------------------
# 5. PPO Training Utilities
# --------------------------------------------------------------
def compute_gae(rewards, values, dones, next_value):
    values = values + [next_value]
    gae = 0
    returns = []
    for step in reversed(range(len(rewards))):
        delta = rewards[step] + GAMMA * values[step + 1] * (1 - dones[step]) - values[step]
        gae = delta + GAMMA * GAE_LAMBDA * (1 - dones[step]) * gae
        returns.insert(0, gae + values[step])
    return returns

# --------------------------------------------------------------
# 6. Main PPO Training Loop
# --------------------------------------------------------------
episode_rewards = []
all_returns = []

print("\nStarting Single-Agent PPO Training...")

global_step = 0
for episode in range(1, NUM_EPISODES + 1):
    state = env.reset()
    episode_reward = 0
    done = False

    # Collect trajectory
    states, actions, rewards, log_probs, values, dones = [], [], [], [], [], []

    while not done:
        state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            logits, value = model(state_t)
            dist = Categorical(logits=logits)
            action = dist.sample()
            log_prob = dist.log_prob(action)

        next_state, reward, done = env.step(action.item())

        states.append(state)
        actions.append(action.item())
        rewards.append(reward)
        log_probs.append(log_prob.item())
        values.append(value.item())
        dones.append(done)

        state = next_state
        episode_reward += reward
        global_step += 1

    # Compute returns & advantages
    with torch.no_grad():
        next_state_t = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        _, next_value = model(next_state_t)
        next_value = next_value.item()

    returns = compute_gae(rewards, values, dones, next_value)
    advantages = np.array(returns) - np.array(values)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # Convert to tensors
    states_t = torch.tensor(np.array(states), dtype=torch.float32, device=device)
    actions_t = torch.tensor(actions, dtype=torch.long, device=device)
    old_log_probs_t = torch.tensor(log_probs, dtype=torch.float32, device=device)
    returns_t = torch.tensor(returns, dtype=torch.float32, device=device)
    advantages_t = torch.tensor(advantages, dtype=torch.float32, device=device)

    # PPO Update
    for _ in range(EPOCHS_PER_UPDATE):
        logits, values_pred = model(states_t)
        dist = Categorical(logits=logits)
        new_log_probs = dist.log_prob(actions_t)
        entropy = dist.entropy().mean()

        ratio = torch.exp(new_log_probs - old_log_probs_t)
        surr1 = ratio * advantages_t
        surr2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * advantages_t
        policy_loss = -torch.min(surr1, surr2).mean()
        value_loss = F.mse_loss(values_pred, returns_t)
        loss = policy_loss + 0.5 * value_loss - 0.01 * entropy

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

    episode_rewards.append(episode_reward)

    if episode % 10 == 0 or episode == 1:
        avg_rew = np.mean(episode_rewards[-10:])
        print(f"Episode {episode:3d}/{NUM_EPISODES} | Avg Reward: {avg_rew:8.2f}")

# --------------------------------------------------------------
# 7. Save Model & Results
# --------------------------------------------------------------
torch.save(model.state_dict(), RESULTS / "single_agent_ppo.pt")
np.save(RESULTS / "ppo_episode_rewards.npy", np.array(episode_rewards))

print(f"\n✓ Single-Agent PPO model saved → results/single_agent_ppo.pt")

# Learning curve
plt.figure(figsize=(10, 5))
plt.plot(episode_rewards, alpha=0.6, label="Episode Reward")
plt.plot(pd.Series(episode_rewards).rolling(10).mean(), color="red", label="Moving Avg (10)")
plt.title("Single-Agent PPO – Credit Decisioning")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.legend()
plt.grid(True)
plt.savefig(RESULTS / "ppo_learning_curve.png", dpi=140, bbox_inches="tight")
plt.show()

print("\n✅ 06_Single_Agent_PPO completed.")
print("Next → 07_Multi_Agent_Training.ipynb (Core of RQ1)")